# 02 — Fit the Jacobian lens and run the Table 3 validation battery

**Stage:** Proposal Stage 2. **Gate:** `Expected_Tables_and_Figures` §3, Table 3 — five
pass/fail checks on the *base model*, before AHN is involved at all. That document is
explicit: *"If either fails, the correct action is to fix the instrumentation, not to
report the RQ tables with a caveat."*

### Where the pilot stands

`fitted-jlens` fitted layer 18 from **one 106-character prompt** at `max_seq_len=64`.
The corpus-averaging step is the entire reason to prefer a J-lens over a logit lens —
the proposal says so in as many words: *"The averaging step is what separates
verbalizable content from content that merely happens to be verbalised in one context."*
A one-prompt map is a single-context Jacobian.

The evidence that it isn't working is already in `AHN_algoverse.ipynb` cell 24: decoding
the **full layer-18 residual stream** through J18 returns `<|endoftext|>`, `小镇`,
`县公安局`, `ABCDE`. The full residual at layer 18 should decode to something coherent —
that is the lens's easiest possible input. It is a fail, and it happened before any AHN
vector was decoded.

**Run Table 3 first. Nothing downstream is interpretable until it passes.**


In [1]:
import os

os.environ["HF_HOME"] = "/home/jupyter-dphs-da30/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/home/jupyter-dphs-da30/hf_cache/hub"
os.environ["HF_DATASETS_CACHE"] = "/home/jupyter-dphs-da30/hf_cache/datasets"

In [ ]:

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-da30/hannah/AHN
working directory pinned to /home/jupyter-dphs-da30/hannah/AHN


In [4]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [5]:
# The J-lens map is a property of the BACKBONE, not of the AHN checkpoint.
# Fit it once on Qwen2.5-3B-Instruct and reuse it for all three cells.
JCFG = dict(
    backbone      = "Qwen/Qwen2.5-3B-Instruct",
    source_layers = [9, 18, 27],   # stratified: early / middle / late
    n_prompts     = 1000,           # proposal Table 3 stability check needs two x 500
    max_seq_len   = 256,
    skip_first    = 4,
    dim_batch     = 8,
   out_path = os.path.join(CFG["results_dir"],"jlens_qwen25_3b_1000ctx.pt"),
)
print(json.dumps(JCFG, indent=2))


{
  "backbone": "Qwen/Qwen2.5-3B-Instruct",
  "source_layers": [
    9,
    18,
    27
  ],
  "n_prompts": 1000,
  "max_seq_len": 256,
  "skip_first": 4,
  "dim_batch": 8,
  "out_path": "results/run_3b_gdn/jlens_qwen25_3b_1000ctx.pt"
}


## The averaging corpus

Must be **strictly disjoint from every evaluation set** (proposal, Datasets section) so
the map cannot encode anything about the test items. Two disjoint halves so Table 3's
map-stability row can actually be computed.


In [6]:
from datasets import load_dataset

def build_corpus(n, seed=ai.SEED, skip=0):
    """Generic English text, disjoint from RULER / LongBench / LV-Eval."""
    # HF migrated bare "wikitext" to the namespaced "Salesforce/wikitext" repo; recent
    # huggingface_hub/datasets versions reject single-segment repo ids outright.
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
    out = []
    for i, ex in enumerate(ds):
        if i < skip:
            continue
        t = ex["text"].strip()
        if len(t) > 400:
            out.append(t[:1500])
        if len(out) >= n:
            break
    return out

corpus_a = build_corpus(JCFG["n_prompts"], skip=0)
corpus_b = build_corpus(JCFG["n_prompts"], skip=20000)   # disjoint half
print(f"corpus A: {len(corpus_a)}   corpus B: {len(corpus_b)}")
print("overlap:", len(set(corpus_a) & set(corpus_b)), "(must be 0)")
assert not (set(corpus_a) & set(corpus_b))


/home/jupyter-dphs-da30/jlens-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


corpus A: 1000   corpus B: 1000
overlap: 0 (must be 0)


## Fit

`jlens` needs `transformers >= 5.x`, which conflicts with the AHN repo's pin
(`transformers==4.51.0`). **Fit in a separate environment / separate session**, save the
`.pt`, and load it in notebook 04. That is why fitting and use are split across
notebooks — it is the version conflict Son hit, handled rather than worked around.

Budget check from the proposal: 15–20 GPU-hours expected, **abort RQ2 above ~40 h**.
The wall-clock printed below is Table 10 row 1. Record it.


In [7]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
btok = AutoTokenizer.from_pretrained(JCFG["backbone"])
bmodel = AutoModelForCausalLM.from_pretrained(
    JCFG["backbone"], torch_dtype=torch.bfloat16, device_map="auto"
).eval()

base = ai.ModelBundle(
    model=bmodel, tokenizer=btok, n_layers=bmodel.config.num_hidden_layers,
    hidden=bmodel.config.hidden_size, vocab=bmodel.lm_head.weight.shape[0],
    num_heads=bmodel.config.num_attention_heads,
    head_dim=getattr(bmodel.config, "head_dim",
                     bmodel.config.hidden_size // bmodel.config.num_attention_heads),
    sliding_window=None, num_attn_sinks=0, use_ahn_router=False, use_q_proj=False,
    use_normalized_l2=False, ahn_layers=[], ahn_impl="none", model_path=JCFG["backbone"],
)
print("backbone loaded", f"{time.time()-t0:.0f}s")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 299.05it/s]


backbone loaded 3s


In [ ]:
t0 = time.time()

ckpt_a = os.path.join(
    CFG["results_dir"],
    "jlens_corpusA_1000ctx.ckpt"
)

lens_a = ai.fit_jacobian_lens(
    base,
    corpus_a,
    JCFG["source_layers"],
    max_seq_len=JCFG["max_seq_len"],
    skip_first=JCFG["skip_first"],
    dim_batch=JCFG["dim_batch"],
    checkpoint_path=ckpt_a,
    checkpoint_every=50,
    resume=True,
)

fit_hours = (time.time() - t0) / 3600
lens_a.meta["fit_gpu_hours"] = fit_hours
lens_a.save(JCFG["out_path"])

print(f"fit corpus A in {fit_hours:.2f} GPU-hours -> {JCFG['out_path']}")

if fit_hours > 40 / len(JCFG["source_layers"]):
    print("! over the proposal's abort threshold — this is the RQ2 go/no-go signal")

/home/jupyter-dphs-da30/jlens-venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:408.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


## Table 3 — the five checks

| # | check | pass criterion |
|---|---|---|
| 1 | final-layer identity | lens at final layer reproduces the model's own next-token distribution, KL < 0.01 |
| 2 | logit-lens agreement | top-1 agreement with logit lens at mid layers ≥ 60% |
| 3 | known-fact recall | "The capital of France is" ranks " Paris" top-1 from ~layer 20 |
| 4 | map stability | two maps from disjoint 500-context corpora agree on top-10 ≥ 80% |
| 5 | map cost | wall-clock GPU-hours (gates the 7B decision, Table 10) |

Check 3 is the one to look at first — it is cheap, and it is the check the pilot's
layer-18 decode implicitly failed.


In [ ]:
import torch

@torch.no_grad()
def resid_at(model, tokenizer, prompt, layer, pos=-1):
    store = {}
    h = model.model.layers[layer].register_forward_hook(
        lambda m, i, o: store.__setitem__("h", (o[0] if isinstance(o, tuple) else o).detach())
    )
    try:
        ins = tokenizer(prompt, return_tensors="pt").to(model.device)
        model(**ins)
    finally:
        h.remove()
    return store["h"][0, pos].float()

T3 = {}

# --- check 3: known-fact recall ---------------------------------------------------
prompt = "The capital of France is"
paris = btok.encode(" Paris", add_special_tokens=False)[0]
rows = []
for L in JCFG["source_layers"]:
    v = resid_at(bmodel, btok, prompt, L)
    lg = ai.readout_logits(v, base, lens=lens_a, layer=L)
    rows.append({"layer": L, "rank_Paris": ai.token_rank(lg, paris),
                 "top5": [btok.decode([int(i)]) for i in lg.topk(5).indices]})
    print(rows[-1])
T3["known_fact_recall"] = {
    "rows": rows,
    "passed": any(r["rank_Paris"] == 0 for r in rows if r["layer"] >= 18),
}
print("\ncheck 3:", "PASS" if T3["known_fact_recall"]["passed"] else "FAIL")


In [ ]:
# --- check 2: agreement with the plain logit lens ---------------------------------
import numpy as np
probe_prompts = corpus_b[:40]
agree = {L: [] for L in JCFG["source_layers"]}
for p in probe_prompts:
    for L in JCFG["source_layers"]:
        v = resid_at(bmodel, btok, p[:800], L)
        j = ai.readout_logits(v, base, lens=lens_a, layer=L)
        l = ai.readout_logits(v, base, lens=None)
        agree[L].append(float(int(j.argmax()) == int(l.argmax())))
T3["logit_lens_agreement"] = {str(L): float(np.mean(a)) for L, a in agree.items()}
mid = JCFG["source_layers"][len(JCFG["source_layers"]) // 2]
T3["logit_lens_agreement"]["passed"] = bool(np.mean(agree[mid]) >= 0.60)
print(json.dumps(T3["logit_lens_agreement"], indent=2))


In [ ]:
# --- check 1: final-layer identity ------------------------------------------------
# The lens at the final layer should be (near) the identity: transporting the last
# residual and decoding must reproduce the model's own next-token distribution.
final_layer = base.n_layers - 1
if final_layer in lens_a.jacobians:
    p_txt = corpus_b[0][:600]
    ins = btok(p_txt, return_tensors="pt").to(bmodel.device)
    with torch.no_grad():
        true_logits = bmodel(**ins).logits[0, -1].float()
    v = resid_at(bmodel, btok, p_txt, final_layer)
    lens_logits = ai.readout_logits(v, base, lens=lens_a, layer=final_layer)
    P = torch.softmax(true_logits, -1); Q = torch.softmax(lens_logits, -1)
    kl = float((P * (P.clamp_min(1e-12).log() - Q.clamp_min(1e-12).log())).sum())
    T3["final_layer_identity"] = {"kl": kl, "passed": kl < 0.01}
else:
    T3["final_layer_identity"] = {
        "skipped": True,
        "note": f"layer {final_layer} not in source_layers; add it to run this check",
    }
print(json.dumps(T3["final_layer_identity"], indent=2))


In [ ]:
# --- check 4: map stability across disjoint corpora --------------------------------
# The expensive one -- same cost as fitting lens_a, ~1.9 GPU-h at n_prompts=500 per the
# corpus_a fit above. This was interrupted once already (KeyboardInterrupt mid-backward-
# pass, no partial map recovered) -- two changes here so that doesn't cost the run again:
#
#   1. checkpoint_path/checkpoint_every/resume flow straight through fit_jacobian_lens's
#      **kwargs into jlens.fit (confirmed in its signature from the interrupt traceback).
#      An interrupt now resumes from the last checkpoint instead of restarting at prompt 0.
#   2. lens_a is reloaded from disk if it isn't already in memory, so this cell also works
#      after a kernel restart (the likely response to an accidental long-running cell) --
#      it does not require re-running cells 1-12 in the same session.
#
# Expect roughly the corpus_a fit time again (~1.9-2.1 GPU-h here). Do NOT interrupt --
# if it needs to stop, use the checkpoint (see below) rather than a KeyboardInterrupt,
# since only the checkpoint path is guaranteed resumable.

import time

if 'lens_a' not in dir() or lens_a is None:
    print("lens_a not in memory -- reloading from", JCFG["out_path"])
    lens_a = ai.JacobianLens.load(JCFG["out_path"], map_location=str(bmodel.device))

_prior_fit_h = lens_a.meta.get("fit_gpu_hours")
if _prior_fit_h:
    print(f"corpus_a took {_prior_fit_h:.2f} GPU-h for {JCFG['n_prompts']} prompts -- "
          f"budget about the same for corpus_b. This is a single blocking call with no "
          f"progress printout apart from checkpoint saves; that is expected, not a hang.")

ckpt_path = os.path.join(CFG["results_dir"], "jlens_corpusB.ckpt")
t0 = time.time()
lens_b = ai.fit_jacobian_lens(
    base, corpus_b, JCFG["source_layers"],
    max_seq_len=JCFG["max_seq_len"], skip_first=JCFG["skip_first"],
    dim_batch=JCFG["dim_batch"],
    checkpoint_path=ckpt_path, checkpoint_every=50, resume=True,
)
fit_b_hours = (time.time() - t0) / 3600
lens_b.meta["fit_gpu_hours"] = fit_b_hours
lens_b.save(os.path.join(CFG["results_dir"], "jlens_qwen25_3b_corpusB.pt"))
print(f"fit corpus B in {fit_b_hours:.2f} GPU-hours")

overlaps = {}
for L in JCFG["source_layers"]:
    hits = []
    for p in corpus_b[:30]:
        v = resid_at(bmodel, btok, p[:800], L)
        ta = set(int(i) for i in ai.readout_logits(v, base, lens=lens_a, layer=L).topk(10).indices)
        tb = set(int(i) for i in ai.readout_logits(v, base, lens=lens_b, layer=L).topk(10).indices)
        hits.append(len(ta & tb) / 10.0)
    overlaps[str(L)] = float(np.mean(hits))
T3 = globals().get("T3", {})
T3["map_stability"] = {**overlaps, "passed": bool(min(overlaps.values()) >= 0.80),
                       "fit_b_gpu_hours": fit_b_hours, "checkpoint_path": ckpt_path}
print(json.dumps(T3["map_stability"], indent=2))
print()
print("Reading this number: >=0.80 at every layer means the map has converged -- a weak")
print("or backwards signal in notebook 04's control battery is then a real representational")
print("limitation, not a fitting artifact, which is stronger evidence for dropping to RQ1.")
print("Below 0.80 at any layer means the map hasn't converged -- more contexts could still")
print("rescue RQ2, which is a materially different message to bring to Gautam.")


In [ ]:
# Merge with whatever is already saved on disk rather than overwrite it -- if this
# session only ran cell 13 (e.g. after a kernel restart) and checks 1-3 (T3 keys
# "known_fact_recall", "logit_lens_agreement", "final_layer_identity") only exist in a
# PREVIOUS session's saved file, clobbering here would silently lose them.
_prior_path = "02_table3_jlens_validation.json"
try:
    _prior = ai.load_json(_prior_path)
    print(f"found existing {_prior_path} from a prior run -- merging, not overwriting")
except FileNotFoundError:
    _prior = {}

T3_saved = {**_prior, **T3}   # this session's in-memory results win on key conflicts
T3_saved["map_cost_gpu_hours"] = fit_hours if 'fit_hours' in dir() else _prior.get("map_cost_gpu_hours")
T3_saved["config"] = JCFG

required = ["known_fact_recall", "logit_lens_agreement", "map_stability"]
missing = [k for k in required if k not in T3_saved]
if missing:
    print(f"! cannot compute TABLE_3_PASSED yet -- missing: {missing}")
    print("  (run cells 10-12 in this session, or confirm they're in the prior saved file)")
    T3_saved["TABLE_3_PASSED"] = None
else:
    T3_saved["TABLE_3_PASSED"] = all(T3_saved[k].get("passed", False) for k in required)

ai.save_json(T3_saved, _prior_path)
print(f"saved -> {os.path.join(CFG['results_dir'], _prior_path)}")
print(json.dumps({k: (v.get("passed") if isinstance(v, dict) else v)
                  for k, v in T3_saved.items()}, indent=2))
print()
status = T3_saved["TABLE_3_PASSED"]
print("TABLE 3:", "PASS" if status else ("INCOMPLETE" if status is None else
                                          "FAIL — fix the lens, do not proceed"))


## Tests Post Check 1 & 2 Failed

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.mem_get_info(0)[0] / 1e9, "GB free")

In [ ]:
p_txt = "The capital of France is"
ins = btok(p_txt, return_tensors="pt").to(bmodel.device)
with torch.no_grad():
    true_logits = bmodel(**ins).logits[0, -1].float()

v_last = resid_at(bmodel, btok, p_txt, base.n_layers - 1)   # layer 35
mine   = ai.readout_logits(v_last, base, lens=None)          # final_norm -> lm_head

P, Q = torch.softmax(true_logits, -1), torch.softmax(mine, -1)
kl = float((P * (P.clamp_min(1e-12).log() - Q.clamp_min(1e-12).log())).sum())
print("KL(model || our decode path) =", kl, "  <- should be ~0")
print("model top5:", [btok.decode([int(i)]) for i in true_logits.topk(5).indices])
print("ours  top5:", [btok.decode([int(i)]) for i in mine.topk(5).indices])

In [ ]:
resid_cache = {}
def resid_c(L):
    if L not in resid_cache:
        resid_cache[L] = resid_at(bmodel, btok, p_txt, L)
    return resid_cache[L]

def decode_with(J, v, transposed):
    t = (v.float() @ J.T) if transposed else (v.float() @ J)
    t = bmodel.model.norm(t.to(bmodel.model.norm.weight.dtype)).float()
    return t @ bmodel.lm_head.weight.float().T

paris = btok.encode(" Paris", add_special_tokens=False)[0]
print(f"{'J':>4} {'resid@':>7} {'orient':>7} {'rank':>7}   top3")
for L_jac in sorted(lens_a.jacobians):
    J = lens_a.jacobians[L_jac].to(bmodel.device)
    for off in (-2, -1, 0, 1, 2):
        L_hook = L_jac + off
        if not (0 <= L_hook < base.n_layers):
            continue
        for tr in (True, False):
            lg = decode_with(J, resid_c(L_hook), tr)
            print(f"{L_jac:>4} {L_hook:>7} {'J@h' if tr else 'h@J':>7} "
                  f"{ai.token_rank(lg, paris):>7}   "
                  f"{[btok.decode([int(i)]) for i in lg.topk(3).indices]}")

In [ ]:
L = 27
base_prompts = corpus_b[:32]          # held out — corpus_b was never fitted
acc = None
for p in base_prompts:
    lg_p = ai.readout_logits(resid_at(bmodel, btok, p[:600], L), base, lens=lens_a, layer=L)
    acc = lg_p if acc is None else acc + lg_p
mean_lg = acc / len(base_prompts)

lg = ai.readout_logits(resid_at(bmodel, btok, "The capital of France is", L), base, lens=lens_a, layer=L)
cen = lg - mean_lg

print("raw      rank:", ai.token_rank(lg,  paris), [btok.decode([int(i)]) for i in lg.topk(5).indices])
print("centered rank:", ai.token_rank(cen, paris), [btok.decode([int(i)]) for i in cen.topk(5).indices])

In [ ]:
import numpy as np
FACTS = [
    ("The capital of France is", " Paris"),   ("The capital of Japan is", " Tokyo"),
    ("The capital of Italy is", " Rome"),     ("The capital of Germany is", " Berlin"),
    ("The capital of Spain is", " Madrid"),   ("The capital of Russia is", " Moscow"),
    ("The largest planet in the solar system is", " Jupiter"),
    ("The author of Hamlet is William", " Shakespeare"),
]
for L in sorted(lens_a.jacobians):
    js, ls = [], []
    for prompt, ans in FACTS:
        ids = btok.encode(ans, add_special_tokens=False)
        if len(ids) != 1:
            continue
        v = resid_at(bmodel, btok, prompt, L)
        js.append(ai.token_rank(ai.readout_logits(v, base, lens=lens_a, layer=L), ids[0]))
        ls.append(ai.token_rank(ai.readout_logits(v, base, lens=None), ids[0]))
    print(f"L{L:2d}  J-lens median {int(np.median(js)):7d}  top1 {sum(x==0 for x in js)}/{len(js)}"
          f"   |  logit-lens median {int(np.median(ls)):7d}  top1 {sum(x==0 for x in ls)}/{len(ls)}")

In [ ]:
w = bmodel.lm_head.weight.float()
for t in ['____', '________', ':**', ' Paris', ' Tokyo']:
    ids = btok.encode(t, add_special_tokens=False)
    if len(ids) == 1: print(f"{t!r:14} ||W||={w[ids[0]].norm().item():.2f}")

### If Table 3 fails

In order of likelihood:

1. **Corpus too small or too short.** Raise `n_prompts` and `max_seq_len` before
   anything else — this is the pilot's failure mode.
2. **`skip_first` too small.** Early positions are dominated by the BOS/template
   prefix and drag the average.
3. **Missing final norm.** `ai.readout_logits(..., apply_final_norm=True)` handles it;
   the pilot's `vec @ unembed.T` did not.
4. **Wrong layer convention.** `jlens` may index the residual *before* vs *after* a
   block differently from a `register_forward_hook` on `model.model.layers[L]`. Test by
   sweeping ±1 and seeing which makes check 3 pass.

Report the outcome to Gautam either way. The proposal has a **hard go/no-go at Week 8**:
if the lens cannot be made to work, RQ2 and RQ3 are dropped and the paper falls back to
RQ1. Knowing that in Week 6 is worth more than a working lens in Week 10.
